In [ ]:
%pip install langchain_ollama langchain_community faiss-gpu
%pip install pandas

In [39]:
from langchain_ollama import ChatOllama
from langchain.agents import tool
from langchain.agents import initialize_agent, Tool, AgentType
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_ollama import ChatOllama, OllamaEmbeddings, OllamaLLM
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotPromptTemplate,
    PromptTemplate,
    SystemMessagePromptTemplate,
    MessagesPlaceholder
)
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.document_loaders import JSONLoader


In [64]:
llm = ChatOllama(
        model="deepseek-v2:lite",
        base_url="https://goose-helped-marmot.ngrok-free.app",
        temperature=0,
        streaming=True,
    )

In [41]:
import pandas as pd

    # Data statis sebagai simulasi hasil eksekusi query SQL
data = {
        'Name': ['Alice', 'Bob', 'Charlie', 'David'],
        'Age': [25, 30, 35, 40],
        'City': ['New York', 'Los Angeles', 'Chicago', 'Houston']
    }

    # Membuat DataFrame
df = pd.DataFrame(data)

In [42]:
agent = create_pandas_dataframe_agent(llm=llm, df=df, verbose=True, allow_dangerous_code=True, handle_parsing_errors=True)

/home/wit.indonesia/Python/be-prompt/.venv/lib/python3.10/site-packages/langchain_experimental/agents/agent_toolkits/pandas/base.py:283: UserWarning: Received additional kwargs {'handle_parsing_errors': True} which are no longer supported.
  warnings.warn(


In [43]:
agent.invoke("halo?")




> Entering new AgentExecutor chain...


ValueError: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Could not parse LLM output: ` Thought: The question "halo?" seems unrelated to the given dataframe and does not provide any context or information.
Action: None`
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 

In [44]:
@tool
def view_chart_from_sql(input: str, query: str, chart_type: str = 'line', color: str = 'blue', title: str = '', xlabel: str = '', ylabel: str = '') -> None:
    """
    Menjalankan query SQL dan menampilkan chart sesuai hasilnya dengan penyesuaian warna dan tipe chart.
    Param Query SQL harus dan wajib relate, relevan dengan input yang sudah disediakan, selain itu tolak saja.
    Minimum limit select 10 count.

    :param input: Input yang diberikan user.
    :param query: Query SQL yang ingin dijalankan.
    :param chart_type: Jenis chart yang ingin digunakan (line, bar, pie).
    :param color: Warna chart.
    :param title: Judul chart.
    :param xlabel: Label untuk sumbu X.
    :param ylabel: Label untuk sumbu Y.
    """
    # Eksekusi query untuk mendapatkan data
    return []

@tool
def view_dataframe_from_sql(query: str) -> pd.DataFrame:
    """
    Menjalankan query SQL dan mengembalikan hasilnya dalam bentuk DataFrame.
    Param Query SQL harus dan wajib relate, relevan dengan input yang sudah disediakan, selain itu tolak saja.
    Minimum limit select 10 count.
    (Simulasi: Data statis digunakan untuk keperluan ini.)

    :param query: Query SQL yang ingin dijalankan.
    :return: DataFrame berisi data hasil eksekusi query.
    """
    import pandas as pd

    # Data statis sebagai simulasi hasil eksekusi query SQL
    data = {
        'Name': ['Alice', 'Bob', 'Charlie', 'David'],
        'Age': [25, 30, 35, 40],
        'City': ['New York', 'Los Angeles', 'Chicago', 'Houston']
    }

    # Membuat DataFrame
    df = pd.DataFrame(data)

    # Return DataFrame (bukan langsung menampilkan, agar lebih fleksibel)
    return df


In [45]:
tools = [view_dataframe_from_sql, view_chart_from_sql]

In [46]:
result = []

for tool in tools:
    result.append({
        "name": tool.name,
        "description": tool.description,
        "args": tool.args
    })

print(result)

[{'name': 'view_dataframe_from_sql', 'description': 'Menjalankan query SQL dan mengembalikan hasilnya dalam bentuk DataFrame.\nParam Query SQL harus dan wajib relate, relevan dengan input yang sudah disediakan, selain itu tolak saja.\nMinimum limit select 10 count.\n(Simulasi: Data statis digunakan untuk keperluan ini.)\n\n:param query: Query SQL yang ingin dijalankan.\n:return: DataFrame berisi data hasil eksekusi query.', 'args': {'query': {'title': 'Query', 'type': 'string'}}}, {'name': 'view_chart_from_sql', 'description': 'Menjalankan query SQL dan menampilkan chart sesuai hasilnya dengan penyesuaian warna dan tipe chart.\nParam Query SQL harus dan wajib relate, relevan dengan input yang sudah disediakan, selain itu tolak saja.\nMinimum limit select 10 count.\n\n:param input: Input yang diberikan user.\n:param query: Query SQL yang ingin dijalankan.\n:param chart_type: Jenis chart yang ingin digunakan (line, bar, pie).\n:param color: Warna chart.\n:param title: Judul chart.\n:para

In [ ]:
loader = JSONLoader(
    file_path='./example_data/facebook_chat.json',
    jq_schema='.messages[].content',
    text_content=False)

data = loader.load()

In [47]:
examples = [
    {"question": "List all artists.", "query": "SELECT * FROM Artist;", "description": ""},
    {"question": "List all cms.", "query": "SELECT * FROM cms;", "description": ""},
    {"question": "List all transactions.", "query": "SELECT * FROM transaction_schema.transactions limit 10;", "description": ""},
    {"question": "List all last month.", "query": "SELECT * FROM last_month;", "description": ""},
    {"question": "Top Product Therapist.", "query": "SELECT report_schema.report_top_product_therapist('5cec917d-2aab-48bb-923b-8642a8a3c1cc', false, null , null);", "description": ""}
]

system_prefix = """Anda adalah asisten AI yang sangat hebat, pintar, terampil, dan cerdas dalam menjawab pertanyaan. Anda dapat menggunakan tools berikut sesuai kebutuhan:

             {tools_description_text}

            Semua respons Anda sangat **harus** dalam format JSON berikut:
            {{
              "tool_calls": [
                {{
                  "id": "id_value",
                  "function": {{
                    "arguments": "{{\\"arg_name\\": \\"arg_value\\"}}",
                    "name": "tool_name"
                  }},
                  "type": "function"
                }}
              ],
              "message": "Jika pertanyaan tidak relevan dengan tools, beri tahu bahwa pertanyaan tersebut tidak relevan dan berikan saran agar sesuai."
            }}

            - Jika input pengguna sesuai dengan alat, gunakan alat tersebut untuk memberikan jawaban.
            - Jika input pengguna tidak sesuai dengan kumpulan input yang ada maka jangan pakai tools.
            - Jika input pengguna tidak relevan dengan alat yang tersedia, beri tahu pengguna bahwa pertanyaan tersebut tidak relevan dengan alat yang ada dan sarankan agar pertanyaan tersebut bisa lebih spesifik agar sesuai dengan alat yang tersedia.
            """

# example_selector = SemanticSimilarityExampleSelector.from_examples(
#     examples,
#     OllamaEmbeddings(model='nomic-embed-text'),
#     FAISS,
#     k=1,
#     input_keys=["input"],
# )

# few_shot_prompt = FewShotPromptTemplate(
#     example_selector=example_selector,
#     example_prompt=PromptTemplate.from_template(
#         "{query}"
#     ),
#     input_variables=["input"],
#     prefix=system_prefix,
#     suffix="",
# )

# full_prompt = ChatPromptTemplate.from_messages(
#     [
#         SystemMessagePromptTemplate(prompt=few_shot_prompt),
#         # MessagesPlaceholder(variable_name="history"),
#         ("human", "{input}"),
#     ]
# )

In [9]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Anda adalah asisten AI yang sangat hebat, pintar, terampil, dan cerdas dalam menjawab pertanyaan. Anda dapat menggunakan tools berikut sesuai kebutuhan:

             {tools_description_text}

            Semua respons Anda **harus** dalam format JSON berikut:
            {{
              "tool_calls": [
                {{
                  "id": "id_value",
                  "function": {{
                    "arguments": "{{\\"arg_name\\": \\"arg_value\\"}}",
                    "name": "tool_name"
                  }},
                  "type": "function"
                }}
              ],
              "message": "Jika pertanyaan tidak relevan dengan tools, beri tahu bahwa pertanyaan tersebut tidak relevan dan berikan saran agar sesuai."
            }}

            - Jika input pengguna sesuai dengan alat, gunakan alat tersebut untuk memberikan jawaban.
            - Jika pertanyaan tidak ada di input maka jangan pernah gunakan tool apapun.
            - Jika input pengguna tidak relevan dengan alat yang tersedia, beri tahu pengguna bahwa pertanyaan tersebut tidak relevan dengan alat yang ada dan sarankan agar pertanyaan tersebut bisa lebih spesifik agar sesuai dengan alat yang tersedia.

             Berikut pertanyaan dan query yang sudah disediakan:
             {example_query} 
            """
        ),
        (
            "human", 
            """{input}"""
        ),
    ]
)

In [59]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Anda adalah asisten data yang sangat pintar yang bertugas memproses data berdasarkan input pengguna. Anda HANYA boleh menggunakan query SQL yang tersedia di daftar kumpulan query. Jika tidak ada query yang cocok, berikan respons JSON dengan pesan error. Berikan output dalam format JSON.

            Anda memiliki kumpulan query berikut yang dapat digunakan:
            {examples}
            """
        ),
        (
            "human",
            """{input}"""
        ),
    ]
)

In [68]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

examples = [
    {
        "question": "List all artists.",
        "query": "SELECT * FROM Artist;",
        "description": "Retrieve all records from the 'Artist' table."
    },
    {
        "question": "List all cms.",
        "query": "SELECT * FROM cms;",
        "description": "Retrieve all records from the 'cms' table."
    },
    {
        "question": "List all transactions.",
        "query": "SELECT * FROM transaction_schema.transactions LIMIT 10;",
        "description": "Retrieve the first 10 records from the 'transactions' table in the 'transaction_schema'."
    },
    {
        "question": "List all last month.",
        "query": "SELECT * FROM last_month;",
        "description": "Retrieve all records from the 'last_month' table."
    },
    {
        "question": "Top Product Therapist.",
        "query": "SELECT report_schema.report_top_product_therapist('5cec917d-2aab-48bb-923b-8642a8a3c1cc', false, null , null);",
        "description": "Retrieve the top product therapist report for the specified parameters."
    }
]

# Expected Response Template
response_template = {
    "tool_calls": [
        {
            "id": "id_value",
            "function": {
                "arguments": '{"arg_name": "arg_value"}',
                "name": "tool_name"
            },
            "type": "function"
        }
    ],
    "message": "Jika pertanyaan tidak relevan dengan tools, beri tahu bahwa pertanyaan tersebut tidak relevan dan berikan saran agar sesuai."
}

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a smart assistant specialized in SQL queries and data analysis.
            You have the following examples of user questions and their corresponding SQL queries:

            Examples:
            {examples}

            You also have access to the following tools to execute queries and visualize data:

            Tools:
            {tools}

            **Instructions for Processing Questions:**
            1. If the user question matches or closely resembles any example in the list:
               - Identify the corresponding SQL query.
               - Select **only one tool** that is most relevant to the question's intent.
            2. Ensure all arguments for the tool are valid and match the tool's requirements.
            3. Must be in accordance with the templates.
            3. Tool name harus ada.
            4. Respond **ONLY** in the following JSON format:

            Response Format:
            {response_template}

            **Rules:**
            - Default show, use the tool `view_dataframe_from_sql`.
            - Never use more than one tool for a single question.
            - If the question is unclear or irrelevant, respond politely without calling any tools.
            - Do not add text or explanations outside of this JSON format.
            """
        ),
        MessagesPlaceholder(variable_name="history"),
        (
            "user", 
            """{input}"""
        ),
    ]
)


In [55]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
       (
            "system",
            """Anda adalah asisten AI yang sangat hebat, pintar, terampil dan cerdas dalam menjawab pertanyaan. Anda dapat menggunakan tools berikut sesuai kebutuhan:
            
            - view_dataframe_from_sql: Untuk menampilkan data dalam bentuk tabel berdasarkan query SQL yang relevan dengan pertanyaan.
            - view_chart_from_sql: Untuk membuat visualisasi chart berdasarkan query SQL yang relevan dengan pertanyaan.

            Dataset saat ini:
            {dataset}
            """
        ),
        (
            "user", 
            """{input}"""
        ),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

In [51]:
from langchain.prompts import ChatPromptTemplate
from langchain.prompts.chat import MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
         (
            "system",
            """Anda adalah asisten AI yang hanya memberikan output dalam format JSON. 
            Anda memiliki akses ke tools berikut:
            
            - generate_to_dataframe: Untuk menampilkan data dalam bentuk tabel berdasarkan parameter yang diberikan.
            - generate_to_chart: Untuk membuat visualisasi grafik berdasarkan parameter yang diberikan.

            **Instruksi penting**: Anda harus **hanya** merespons dalam format JSON berikut (tanpa penjelasan lain atau teks tambahan):
            {{
                "tool_name": "[nama_tool]",
                "param": {{[parameter]}}
            }}

            - Gunakan `generate_to_dataframe` jika pengguna meminta data dalam bentuk tabel.
            - Gunakan `generate_to_chart` jika pengguna meminta visualisasi grafik.

            Jangan tambahkan teks atau penjelasan di luar format JSON ini.
            """
        ),
        (
            "user", 
            """{input}"""
        ),
    ]
)


In [22]:
full_prompt.invoke({"input": "haha", "tools_description_text": tools})

NameError: name 'full_prompt' is not defined

In [69]:
runnable = prompt | llm

In [70]:
store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


with_message_history = RunnableWithMessageHistory(
    runnable,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)



In [73]:
with_message_history.invoke(
    {"input": "show me json only", "tools": result, "examples": examples, "response_template":response_template},
    config={"configurable": {"session_id": "40"}},
)

AIMessage(content=' Certainly! Here\'s the response formatted in JSON:\n```json\n{\n    "tool_calls": [\n        {\n            "id": "view_chart_from_sql",\n            "function": {\n                "arguments": "{\\"input\\": {\\"title\\": \\"Input\\", \\"type\\": \\"string\\"}, \\"query\\": {\\"title\\": \\"Query\\", \\"type\\": \\"string\\"}",\n                "args": {"input": {"title": "Input", "type": "string"}, "query": {"title": "Query", "type": "string"}, "chart_type": {"default": \'line\', "title": \'Chart Type\', "type": \'string\'}, "color": {"default": \'blue\', "title": \'Color\', "type": \'string\'}, "title": {"default": \'\', "title": \'Title\', "type": \'string\'}, "xlabel": {"default": \'\', "title": \'Xlabel\', "type": \'string\'}, "ylabel": {"default": \'\', "title": \'Ylabel\', "type": \'string\'}},\n                "description": "Menjalankan query SQL dan menampilkan chart sesuai hasilnya dengan penyesuaian warna dan tipe chart."\n            },\n            "t

In [67]:
chain = prompt | llm
chain.invoke({"input": "list transaction", "tools": result, "examples": examples, "response_template":response_template})


AIMessage(content=' {\n    "tool_calls": [\n        {\n            "id": "1",\n            "function": {\n                "arguments": "{\'query\': {\'title\': \'Query\', \'type\': \'string\'}}",\n                "name": "view_dataframe_from_sql",\n                "args": {"query": {"title": "Query", "type": "string"}}\n            }\n        }\n    ],\n    "message": "Silakan berikan pertanyaan yang lebih spesifik atau jelas, seperti \'list all transactions\' atau \'tampilkan semua transaksi\'. Kecuali Anda memberikan pertanyaan yang lebih spesifik, tidak ada tools yang dapat membantu."\n}', additional_kwargs={}, response_metadata={'model': 'deepseek-v2:lite', 'created_at': '2025-01-28T15:24:22.198376261Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3261566197, 'load_duration': 16503771, 'prompt_eval_count': 1086, 'prompt_eval_duration': 104000000, 'eval_count': 176, 'eval_duration': 3128000000, 'message': Message(role='assistant', content='', images=None, tool_calls=None)

In [7]:
llm_with_tools = llm.bind_tools(tools)

In [7]:
from langchain.agents.format_scratchpad.openai_tools import (
    format_to_openai_tool_messages,
)
from langchain.agents.output_parsers.openai_tools import OpenAIToolsAgentOutputParser

agent = (
    {
        "input": lambda x: x["input"],
        "dataset": lambda x: x["dataset"],
        "agent_scratchpad": lambda x: format_to_openai_tool_messages(
            x["intermediate_steps"]
        ),
    }
    | prompt
    | llm_with_tools
    | OpenAIToolsAgentOutputParser()
)

In [8]:
from langchain.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

NameError: name 'tools' is not defined

In [11]:
list(agent_executor.stream({"input": "haha", "dataset": ""}))





> Entering new None chain...


ResponseError: registry.ollama.ai/library/deepseek-r1:latest does not support tools

In [ ]:
llm_with_tools.invoke("berapa huruf abcd")